# Notebook 3 – 03_Bronze_to_Silver

### Objective

#### This notebook cleans the raw Bronze data.
#### The Silver layer improves data quality by handling missing values, removing duplicate records, fixing formatting issues, correcting data types, and validating business rules.

## Step 1 : Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Step 2 : Configuration

In [0]:
silver_path = "/Volumes/dbacademy/default/myvolume/silver"
silver_table = "silver_upi_transactions"
bronze_table = "bronze_upi_transactions"

## Step 3 : Read Bronze Table

In [0]:
bronze_df = spark.table(bronze_table)

display(bronze_df.limit(10))

transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason,load_timestamp,source_file
2bbc7344-a5dc-4ffb-bff0-171c1c7620c0,2026-07-08T14:04:52.000Z,HDFC,SBI,ankit557@okhdfc,rohit684@oksbi,7090,P2P,SUCCESS,Indore,Madhya Pradesh,DEV100111,2366,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
61f432e0-bd55-4cfe-b14d-2beec8fd638f,2026-06-15T06:45:32.000Z,ICICI,HDFC,priya403@icici,manish738@okhdfc,15597,P2P,SUCCESS,Mumbai,Maharashtra,DEV100174,1016,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
bb370089-0c45-4264-bc56-15f741cbef38,2026-06-25T21:59:21.000Z,ICICI,SBI,manish200@icici,ankit279@oksbi,40824,Merchant,SUCCESS,Jaipur,Rajasthan,DEV100122,217,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
62fb68f6-15e6-4ee9-9478-f772f83d9e33,2026-06-18T15:32:26.000Z,ICICI,HDFC,neha849@icici,arjun756@okhdfc,18156,Recharge,SUCCESS,Hyderabad,Telangana,DEV100003,473,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
79b0fb85-a310-48a2-957b-92e38f6e789f,2026-07-06T08:11:01.000Z,HDFC,Axis,rohit687@okhdfc,arjun923@axis,24257,P2P,SUCCESS,Pune,Maharashtra,DEV100140,2425,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
e0edde04-9c30-43aa-8538-c800a9a3c1ff,2026-06-30T21:33:28.000Z,Axis,HDFC,arjun816@axis,pooja764@okhdfc,33603,P2P,SUCCESS,Bangalore,Karnataka,DEV100014,1272,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
c0de5084-2cb8-412b-976e-79202956f294,2026-07-06T02:01:28.000Z,SBI,Axis,kartik629@oksbi,divya195@axis,14605,Bill Payment,SUCCESS,Pune,Maharashtra,DEV100034,1279,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
3069fc99-7521-402d-8ca6-a10ecdb5cd87,2026-07-04T06:39:40.000Z,HDFC,Axis,pooja190@okhdfc,priya798@axis,17898,Recharge,SUCCESS,Indore,Madhya Pradesh,DEV100174,1996,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
95276c83-5a4c-47b5-96f8-02fa3897504c,2026-06-14T14:36:28.000Z,Axis,SBI,karan234@axis,kartik537@oksbi,21863,P2P,SUCCESS,Hyderabad,Telangana,DEV100193,989,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
eefed37e-8689-43d8-a13a-bbca7b1e841f,2026-06-17T15:31:51.000Z,Axis,ICICI,riya328@axis,prachi153@icici,null,Recharge,SUCCESS,Mumbai,Maharashtra,DEV100088,469,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv


#### 3.1 Rows & Columns

In [0]:
print("Number of rows are: ", bronze_df.count())
print("Number of columns are: ", len(bronze_df.columns))

Number of rows are:  20400
Number of columns are:  16


#### 3.2 Columns Name

In [0]:
print(bronze_df.columns)

['transaction_id', 'transaction_timestamp', 'sender_bank', 'receiver_bank', 'sender_upi', 'receiver_upi', 'amount', 'transaction_type', 'transaction_status', 'city', 'state', 'device_id', 'response_time_ms', 'failure_reason', 'load_timestamp', 'source_file']


### 3.3 Schema

In [0]:
bronze_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- sender_bank: string (nullable = true)
 |-- receiver_bank: string (nullable = true)
 |-- sender_upi: string (nullable = true)
 |-- receiver_upi: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- response_time_ms: integer (nullable = true)
 |-- failure_reason: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



#### 3.4 Check Null Values

In [0]:
null_values = bronze_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in bronze_df.columns
])

display(null_values)

transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason,load_timestamp,source_file
0,0,0,0,0,0,557,0,0,0,0,0,0,19877,0,0


#### 3.5 Duplicate Records Count

In [0]:
duplicate_count = bronze_df.count() - bronze_df.dropDuplicates().count()

print("Duplicate Records :", duplicate_count)

Duplicate Records : 400


#### 3.6 Negative Records Count

In [0]:
print("Number of Negative Records are: ")
bronze_df.filter(bronze_df["amount"] < 0).count()

Number of Negative Records are: 


384

## Step 4 : Handle Null Values

In [0]:
silver_df = bronze_df.dropna(subset=["amount"])

In [0]:
print("After handling null values:")
null_values = silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
])

display(null_values)

After handling null values:


transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason,load_timestamp,source_file
0,0,0,0,0,0,0,0,0,0,0,0,0,19335,0,0


## Step 5 : Remove Duplicate Records

#### 5.1 Check Duplicate Count Before Cleaning

In [0]:
duplicate_count = silver_df.count() - silver_df.dropDuplicates().count()

print(f"Duplicate Records Before Cleaning : {duplicate_count}")

Duplicate Records Before Cleaning : 387


#### 5.2 Remove Duplicates

In [0]:
silver_df = silver_df.dropDuplicates()

#### 5.3 Verify Duplicate Count After Cleaning

In [0]:
duplicate_count = silver_df.count() - silver_df.dropDuplicates().count()

print(f"Duplicate Records After Cleaning : {duplicate_count}")

Duplicate Records After Cleaning : 0


## Step 6 : Trim Extra Spaces

### 6.1 : Identify String Columns

In [0]:
string_columns = ["sender_upi","receiver_upi","sender_bank","receiver_bank","transaction_type","transaction_status",
"city","state","failure_reason"]

### 6.2 : Apply Trim

In [0]:
for column in string_columns:
    silver_df = silver_df.withColumn(column,trim(col(column)))

### 6.3 : Verify

In [0]:
display(silver_df.select("sender_bank").distinct())

sender_bank
HDFC
ICICI
Axis
SBI


In [0]:
display(silver_df.select("city").distinct())

city
Indore
Mumbai
Jaipur
Hyderabad
Pune
Bangalore
Ahmedabad
Delhi


## Step 7: Fix Data Types

### 7.1 Convert Timestamp

In [0]:
silver_df = silver_df.withColumn("transaction_timestamp", 
                                 to_timestamp(col("transaction_timestamp"),"yyyy-MM-dd HH:mm:ss"))

### 7.2 Convert Amount

In [0]:
silver_df = silver_df.withColumn("amount",col("amount").cast("double"))

### 7.3 Convert Response Time

In [0]:
silver_df = silver_df.withColumn("response_time_ms",col("response_time_ms").cast("integer"))

### 7.4 Verify Schema

In [0]:
silver_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- sender_bank: string (nullable = true)
 |-- receiver_bank: string (nullable = true)
 |-- sender_upi: string (nullable = true)
 |-- receiver_upi: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- response_time_ms: integer (nullable = true)
 |-- failure_reason: string (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



## Step 8 : Remove Invalid Transactions

### 8.1 : Check Invalid Records

In [0]:
invalid_amount_count = silver_df.filter(col("amount")<0).count()

print("Negative Amount Records :" ,invalid_amount_count)

Negative Amount Records : 381


### 8.2 : Remove Negative Amount Records

In [0]:
silver_df = silver_df.filter(col("amount") >= 0)

### 8.3 : Verify

In [0]:
print("Records After Removing Invalid Transactions :")
print(silver_df.count())

Records After Removing Invalid Transactions :
19075


## Step 9 – Standardize Data

### 9.1 : Select Columns to Convert to Uppercase

In [0]:
columns_to_upper = ["sender_bank","receiver_bank","transaction_type","transaction_status","city","state"]

### 9.2 – Apply Uppercase

In [0]:
for column in columns_to_upper:
    silver_df = silver_df.withColumn(column,upper(col(column)))

### 9.3 : Verify

In [0]:
display(silver_df.select("sender_bank").distinct())

sender_bank
HDFC
ICICI
AXIS
SBI


In [0]:
display(silver_df.select("transaction_status").distinct())

transaction_status
SUCCESS
FAILED


In [0]:
display(silver_df.select("transaction_type").distinct())

transaction_type
P2P
MERCHANT
RECHARGE
BILL PAYMENT


## Step 10 : Mask Sensitive Data

In [0]:
def mask_upi(upi):
    if upi is None:
        return None

    username, handle = upi.split("@")

    if len(username) <= 4:
        masked_username = username[0] + "***"
    else:
        masked_username = (username[:2] + "****" + username[-2:])

    return masked_username + "@" + handle

### 10.1 : Register UDF

In [0]:
mask_upi_udf = udf(mask_upi, StringType())

### 10.2 : Mask Sender UPI

In [0]:
silver_df = silver_df.withColumn("sender_upi",mask_upi_udf(col("sender_upi")))

## 10.3 : Mask Receiver UPI

In [0]:
silver_df = silver_df.withColumn("receiver_upi",mask_upi_udf(col("receiver_upi")))

### 10.4 – Verify

In [0]:
masked_data = silver_df.select("sender_upi","receiver_upi")

display(masked_data.limit(5))

sender_upi,receiver_upi
an****30@icici,ka****22@oksbi
ro****66@oksbi,ri****23@axis
ma****95@icici,di****06@axis
pr****73@icici,pr****41@axis
am****44@icici,di****22@oksbi


## Step 11 : Write Silver Layer

In [0]:
silver_df.write \
             .format("delta") \
             .mode("overwrite") \
             .saveAsTable(silver_table)

print("Silver Table Created Successfully")

Silver Table Created Successfully


### 11.1 : Verify Silver Table

In [0]:
s_table = spark.table(silver_table)
display(s_table.limit(5))

transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason,load_timestamp,source_file
29fb4997-5692-4449-a7a3-5ac5a89017e2,2026-07-01T06:51:26.000Z,ICICI,SBI,an****30@icici,ka****22@oksbi,45291.0,P2P,SUCCESS,PUNE,MAHARASHTRA,DEV100030,2613,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
8965c981-9082-4302-80ea-a6143fb8ec8f,2026-07-03T20:42:48.000Z,SBI,AXIS,ro****66@oksbi,ri****23@axis,13260.0,BILL PAYMENT,SUCCESS,BANGALORE,KARNATAKA,DEV100057,441,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
738577eb-907d-411d-a632-b0fd92d700b9,2026-06-27T08:36:52.000Z,ICICI,AXIS,ma****95@icici,di****06@axis,30587.0,RECHARGE,SUCCESS,MUMBAI,MAHARASHTRA,DEV100041,128,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
3770dbc7-ec78-400b-8406-32ed60678044,2026-07-02T17:07:14.000Z,ICICI,AXIS,pr****73@icici,pr****41@axis,25455.0,MERCHANT,SUCCESS,BANGALORE,KARNATAKA,DEV100032,541,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
b1ebfc4b-f6d8-48b9-8a6a-6d5da03bb0ca,2026-07-06T03:02:40.000Z,ICICI,SBI,am****44@icici,di****22@oksbi,33119.0,P2P,FAILED,INDORE,MADHYA PRADESH,DEV100098,2648,Bank Server Down,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv


In [0]:
print("Total Records :", s_table.count())

Total Records : 19075
